In [ ]:
#!/usr/bin/env python3
"""
Multi-objective, multi-concept optimization with PySR + Pyomo + SCIP.

Key numerical conventions:
A) X, F and G are all rounded to 12 decimal places at storage/usage boundaries.
B) All feasibility checks use SCIP's feasibility tolerance `feas_tol` from CONFIG.
C) All "proximity" checks (e.g., distance to archive) use `eps` from CONFIG.
D) Dominance uses constrained ε-dominance with `eps_dom` from CONFIG.

Analysis-failure handling:
1) If a true analysis fails (exception or non-finite F/G), we store NaN in F/G
   for that design point in the archive.
2) Surrogate models for F and G are trained only on rows where the respective
   target column is finite (per-column filtering).
3) If there is at least one NaN in the archive (in any F or G column), we
   train a PySR "classifier" with labels:
       1 -> analysis success (no NaNs in any F or G on that row)
       0 -> analysis failure (some NaN in F or G).
4) The classifier is converted to an additional constraint in the MINLP:
       g_clf(x) = 0.5 - clf_success(x) <= 0
   so that clf_success(x) >= 0.5 is enforced in the Pyomo models
   (both warm-up single-objective and Tchebycheff formulations).

Budget logic (problem-dependent):
Let D_c be the dimension of concept c, and D_total = sum_c D_c.
Config interface:
- lhs_per_dim (default: 10)
    → each concept gets lhs_per_dim * D_c initial LHS samples.
- budget_per_dim (default: 50)
    → global total evaluation budget is budget_per_dim * D_total.
      This includes: LHS + warm-up f1/f2 + all Tchebycheff evals.
"""

import os
import json
import math
import warnings
import random
from pathlib import Path
from functools import reduce
import operator

import numpy as np
import pandas as pd
import sympy as sp
import pyomo.environ as pyo
from pyomo.common.errors import ApplicationError
from scipy.stats import qmc
from pysr import PySRRegressor
import matplotlib.pyplot as plt
import pickle
from joblib import Parallel, delayed
# Import problems and evaluation functions
from Problems import (
    problem_dtlz2,
    problem_zdt3,
    problem_bnh,
    problem_tnk,
    problem_simulator,
    problem_welded_beam,
)

# ----------------------------------------------------
# Environment & warnings
# ----------------------------------------------------
os.environ["DYLD_LIBRARY_PATH"] = "/usr/local/Cellar/gcc/14.2.0_1/lib/gcc/14"
warnings.filterwarnings("ignore", category=UserWarning, module="pysr.sr")

ROUND_DECIMALS = 12
IDEAL_NADIR_PREC = 1e-6   # precision for ideal/nadir & F-quantization


def round_array(arr, decimals=ROUND_DECIMALS):
    """Round numpy array or scalar to the given decimals."""
    return np.round(np.asarray(arr, dtype=float), decimals)


def round_to_precision(arr, prec=IDEAL_NADIR_PREC):
    """
    Quantize array to a given precision, e.g. 1e-6.
    Used for F-values when computing/global ideal & nadir, and for scaling.
    """
    arr = np.asarray(arr, dtype=float)
    return np.round(arr / prec) * prec


def to_2d_rounded(x, decimals=ROUND_DECIMALS):
    """Ensure x is 2D float array and rounded."""
    xv = np.atleast_2d(x).astype(float)
    return np.round(xv, decimals)


# ----------------------------------------------------
# LHS (QMC) sampler
# ----------------------------------------------------
def sample_lhs_qmc(n_samples, dim, lb, ub):
    sampler = qmc.LatinHypercube(d=dim)
    U = sampler.random(n=n_samples)
    return qmc.scale(U, lb, ub)


# ====================================================
# SymPy / PySR helpers
# ====================================================

def sympy_to_pyomo(sym_expr, pyomo_model):
    """Convert a SymPy expression to a Pyomo expression."""
    if isinstance(sym_expr, (sp.Float, sp.Integer)) or sym_expr.is_Number:
        return float(sym_expr)

    if isinstance(sym_expr, sp.Symbol):
        name = str(sym_expr)
        if name.startswith('x'):
            idx = int(name[1:])
            return pyomo_model.x[idx]
        try:
            return float(name)
        except Exception:
            raise NotImplementedError(f"Unhandled sympy symbol '{name}'")

    if isinstance(sym_expr, sp.Add):
        return reduce(operator.add, [sympy_to_pyomo(a, pyomo_model) for a in sym_expr.args])
    if isinstance(sym_expr, sp.Mul):
        return reduce(operator.mul, [sympy_to_pyomo(a, pyomo_model) for a in sym_expr.args])
    if isinstance(sym_expr, sp.Pow):
        b, e = sym_expr.args
        return sympy_to_pyomo(b, pyomo_model) ** sympy_to_pyomo(e, pyomo_model)

    func, args = sym_expr.func, sym_expr.args
    if func == sp.sin:
        return pyo.sin(sympy_to_pyomo(args[0], pyomo_model))
    if func == sp.cos:
        return pyo.cos(sympy_to_pyomo(args[0], pyomo_model))
    if func == sp.exp:
        return pyo.exp(sympy_to_pyomo(args[0], pyomo_model))
    if func == sp.log:
        return pyo.log(sympy_to_pyomo(args[0], pyomo_model))
    if func == sp.sqrt:
        return pyo.sqrt(sympy_to_pyomo(args[0], pyomo_model))
    if func == sp.atan:
        return pyo.atan(sympy_to_pyomo(args[0], pyomo_model))
    if func == sp.atan2:
        y = sympy_to_pyomo(args[0], pyomo_model)
        x = sympy_to_pyomo(args[1], pyomo_model)
        return pyo.atan2(y, x) if hasattr(pyo, "atan2") else pyo.atan(y / (x + 1e-6))

    # last resort: numeric approximation
    try:
        return float(sp.N(sym_expr))
    except Exception:
        raise NotImplementedError(f"Cannot convert sympy expr: {sym_expr}")


def extract_sympy(model):
    """
    Lightweight SymPy extractor from PySR model.

    Now supports a custom chosen equation index (model._chosen_equation_index)
    selected via validation MSE across the nondominated (loss, complexity) front.
    """
    # If we manually selected an equation index, use that first
    if hasattr(model, "_chosen_equation_index"):
        idx = model._chosen_equation_index
        try:
            eqs = model.equations_
            row = eqs.iloc[int(idx)]
            for key in ("sympy_format", "equation"):
                try:
                    val = row.get(key, None)
                except AttributeError:
                    val = None
                if val is not None:
                    return sp.sympify(val)
        except Exception as e:
            print(f"[extract_sympy] Failed to use chosen equation index {idx}: {e}; falling back to get_best().")

    # Fallback: original behavior using get_best()
    best = model.get_best()

    for key in ("sympy_format", "equation"):
        try:
            val = best.get(key, None)
        except AttributeError:
            val = None
        if val is not None:
            return sp.sympify(val)

    if isinstance(best, str):
        return sp.sympify(best)

    raise RuntimeError(f"Could not extract sympy from best model: {best}")


def get_sympy_exprs_from_pysr_models(symbolic_models):
    """Convert a dict of trained PySR models to SymPy expressions."""
    return {k: extract_sympy(m) for k, m in symbolic_models.items()}


def add_classifier_constraint(sympy_exprs):
    """
    If a classifier 'clf_success' is present, add it as an extra constraint:
        g_new(x) = 0.5 - clf_success(x) <= 0
    """
    if "clf_success" not in sympy_exprs:
        return sympy_exprs

    clf_expr = sympy_exprs["clf_success"]
    existing_g = [k for k in sympy_exprs if k.startswith("g")]
    next_idx = len(existing_g) + 1
    new_g_key = f"g{next_idx}"
    sympy_exprs[new_g_key] = 0.5 - clf_expr
    print(f"[add_classifier_constraint] Added classifier-based constraint as '{new_g_key}'.")
    return sympy_exprs


# ====================================================
# Dominance, feasibility & distance utilities
# ====================================================

def normalized_distance_to_archive(x_new, df_X, LB, UB, eps):
    """
    Normalized distance used for proximity checks.
    Uses eps from CONFIG to avoid division by zero and
    as the closeness threshold.
    """
    X = df_X.values if hasattr(df_X, "values") else np.asarray(df_X)
    if X.size == 0:
        return np.array([])
    denom = np.maximum(UB - LB, eps)
    Xn = (X - LB) / denom
    xn = (x_new - LB) / denom
    d = np.linalg.norm(Xn - xn, axis=1)
    return d


def is_feasible_row(g_row, feas_tol):
    """Feasibility check using SCIP feasibility tolerance."""
    arr = np.asarray(g_row, float)
    if arr.size == 0:
        return True
    if np.isnan(arr).any():
        return False
    return np.all(arr <= feas_tol)


def sum_pos_violation(g_row):
    """Sum of positive constraint violations (NaNs treated as large penalty)."""
    arr = np.asarray(g_row, float)
    if arr.size == 0:
        return 0.0
    arr = np.where(np.isnan(arr), 1e6, arr)
    return float(np.sum(np.maximum(0.0, arr)))


def constrained_dominates(Fa, Ga, Fb, Gb, feas_tol, eps_dom=0.0):
    """
    Return True if (Fa, Ga) constrained-dominates (Fb, Gb).

    Rules (Deb-style constrained dominance + ε-dominance):
      1) Feasible vs infeasible:
           - Any feasible point dominates any infeasible point.
      2) Both feasible:
           - Apply ε-dominance in F-space:
               Fa_i <= Fb_i - eps_dom for all i, and
               Fa_j <  Fb_j - eps_dom for some j.
      3) Both infeasible:
           - The one with smaller total positive violation dominates.
    """
    Fa = np.asarray(Fa, float)
    Fb = np.asarray(Fb, float)
    Ga = np.asarray(Ga, float)
    Gb = np.asarray(Gb, float)

    # total positive violation
    viol_a = sum_pos_violation(Ga)
    viol_b = sum_pos_violation(Gb)

    feas_a = viol_a <= feas_tol
    feas_b = viol_b <= feas_tol

    # Case 1: feasibility first
    if feas_a and not feas_b:
        return True
    if feas_b and not feas_a:
        return False

    # Case 2: both feasible → ε-dominance in objectives
    if feas_a and feas_b:
        leq = np.all(Fa <= Fb - eps_dom)
        strict = np.any(Fa < Fb - eps_dom)
        return bool(leq and strict)

    # Case 3: both infeasible → compare violation
    if viol_a < viol_b - feas_tol:
        return True
    return False


def constrained_nondominated_mask(F, G, feas_tol, eps_dom=0.0):
    """
    Constrained nondominated set under constrained_dominates.
    Returns boolean mask for rows of F/G that are nondominated.
    """
    F = np.asarray(F, float)
    if G is None:
        G = np.zeros((F.shape[0], 0))
    G = np.asarray(G, float)

    n = F.shape[0]
    dominated = np.zeros(n, dtype=bool)

    for i in range(n):
        if dominated[i]:
            continue
        for j in range(n):
            if i == j or dominated[i]:
                continue
            if constrained_dominates(F[j], G[j], F[i], G[i], feas_tol, eps_dom):
                dominated[i] = True
                break

    return ~dominated


def is_nondominated(F):
    """
    Standard (unconstrained) Pareto nondominance mask in objective space.
    F: (n_points, n_obj)
    """
    F = np.asarray(F, float)
    n = F.shape[0]
    dominated = np.zeros(n, dtype=bool)
    for i in range(n):
        if dominated[i]:
            continue
        for j in range(n):
            if i == j or dominated[i]:
                continue
            if np.all(F[j] <= F[i]) and np.any(F[j] < F[i]):
                dominated[i] = True
                break
    return ~dominated


def euclidean_dist_matrix(A, B):
    A = np.asarray(A, float)
    B = np.asarray(B, float)
    diff = A[:, None, :] - B[None, :, :]
    return np.sqrt(np.sum(diff * diff, axis=2))


# ====================================================
# Handling analysis failure
# ====================================================

def safe_true_evaluate(eval_func, x, n_obj, n_con):
    """
    Robust wrapper around the 'true' evaluation function.

    Returns:
        F (1D, length n_obj),
        G (1D, length n_con, or length 0 if n_con==0),
        success (bool).
    On failure (exception, size mismatch, or non-finite values) it returns NaNs
    in F and G (except unconstrained problems where G is zero-length) and
    success=False.
    """
    try:
        F_raw, G_raw = eval_func(x)
    except Exception as e:
        F = np.full((n_obj,), np.nan, dtype=float)
        G = np.full((n_con,), np.nan, dtype=float) if n_con > 0 else np.zeros((0,), dtype=float)
        print(f"[safe_true_evaluate] Exception during eval: {e}. Marking as failure.")
        return F, G, False

    F = np.atleast_1d(F_raw).astype(float).flatten()
    if F.size != n_obj:
        print(f"[safe_true_evaluate] F size mismatch ({F.size} != {n_obj}). Marking as failure.")
        F = np.full((n_obj,), np.nan, dtype=float)
        G = np.full((n_con,), np.nan, dtype=float) if n_con > 0 else np.zeros((0,), dtype=float)
        return F, G, False

    if n_con > 0:
        G = np.atleast_1d(G_raw).astype(float).flatten()
        if G.size != n_con:
            print(f"[safe_true_evaluate] G size mismatch ({G.size} != {n_con}). Marking as failure.")
            G = np.full((n_con,), np.nan, dtype=float)
            return F, G, False
    else:
        G = np.zeros((0,), dtype=float)

    finite_F = np.all(np.isfinite(F))
    finite_G = (G.size == 0) or np.all(np.isfinite(G))
    success = finite_F and finite_G

    if not success:
        print("[safe_true_evaluate] Non-finite F/G detected. Marking as failure.")
        if not finite_F:
            F[~np.isfinite(F)] = np.nan
        if G.size > 0 and not finite_G:
            G[~np.isfinite(G)] = np.nan

    return F, G, success


# ====================================================
# PySR model training (regressors + classifier)
# ====================================================

def _train_single_pysr(X, y, iters, rng_seed, label, val_fraction=0.2):
    """
    Train a single PySRRegressor with an 80/20 train/validation split.

    Steps:
    - Filter to finite y.
    - Split into train/val (80/20 by default).
    - Fit PySR on the training subset.
    - From model.equations_, take ALL nondominated equations in
      (train loss, complexity) space.
    - Evaluate each of these on the validation set and compute validation MSE.
    - Select the equation with lowest validation MSE and store its index as
      `model._chosen_equation_index` so that extract_sympy() uses it.

    If a validation set is not available or something fails, we fall back to
    PySR's internal best-equation logic (model.get_best()).
    """
    mask = np.isfinite(y)
    if mask.sum() < 3:
        print(f"[train_symbolic_models] Skipping training for {label}: only {mask.sum()} finite points.")
        return None

    X_clean = X[mask]
    y_clean = y[mask]
    n = X_clean.shape[0]

    rng_args = {} if rng_seed is None else {"random_state": rng_seed}
    model = PySRRegressor(
        niterations=iters,
        population_size=60,
        parsimony=0.001,
        binary_operators=["+", "-", "*", "/"],
        unary_operators=["sin", "cos", "exp", "log", "sqrt"],
        optimizer_iterations=25,
        model_selection="best",  # internal, but we override choice using _chosen_equation_index
        verbosity=0,
        **rng_args,
    )
    model.fit(X_clean, y_clean)

    # If we have a validation set, perform nondominated selection + val MSE
    if X_clean is not None and X_clean.shape[0] > 0:
        try:
            eqs = model.equations_
        except Exception as e:
            eqs = None
            print(f"[train_symbolic_models] {label}: could not access equations_ ({e}); skipping val-based selection.")

        if eqs is not None and len(eqs) > 0 and "loss" in eqs.columns and "complexity" in eqs.columns:
            try:
                # Non-dominated set in (loss, complexity) space (both to be minimized)
                metrics = eqs[["loss", "complexity"]].to_numpy(dtype=float)
                nd_mask = is_nondominated(metrics)
                nd_indices = np.where(nd_mask)[0]
                if nd_indices.size == 0:
                    nd_indices = np.arange(len(eqs))

                D = X_clean.shape[1]
                x_syms = sp.symbols(f"x0:{D}")

                best_idx = None
                best_mse = np.inf
                n_evaluated = 0

                for idx in nd_indices:
                    row = eqs.iloc[int(idx)]
                    expr_str = None
                    for key in ("sympy_format", "equation"):
                        try:
                            expr_str = row.get(key, None)
                        except AttributeError:
                            expr_str = None
                        if expr_str is not None:
                            break
                    if expr_str is None:
                        continue

                    try:
                        expr = sp.sympify(expr_str)
                        f_lam = sp.lambdify(x_syms, expr, modules=["numpy"])
                        # Evaluate on validation set
                        Xv = np.asarray(X_clean, dtype=float)
                        args = [Xv[:, j] for j in range(D)]
                        y_pred = f_lam(*args)
                        y_pred = np.asarray(y_pred, dtype=float).reshape(-1)
                        if y_pred.shape[0] != y_clean.shape[0]:
                            continue
                        mse = float(np.mean((y_pred - y_clean) ** 2))
                    except Exception:
                        continue

                    if not np.isfinite(mse):
                        continue

                    n_evaluated += 1
                    if mse < best_mse:
                        best_mse = mse
                        best_idx = int(idx)

                if best_idx is not None:
                    model._chosen_equation_index = best_idx
                    print(
                        f"[train_symbolic_models] {label}: "
                        f"selected equation idx {best_idx} from {len(nd_indices)} ND models "
                        f"based on validation MSE = {best_mse:.4g} "
                        f"(val size {X_clean.shape[0]}, train size {X_clean.shape[0]})"
                    )
                else:
                    print(
                        f"[train_symbolic_models] {label}: "
                        f"could not select a ND equation by validation MSE; "
                        f"falling back to PySR's internal best."
                    )
            except Exception as e:
                print(
                    f"[train_symbolic_models] {label}: error during ND+val selection "
                    f"({e}); falling back to PySR's internal best."
                )
        else:
            print(
                f"[train_symbolic_models] {label}: equations_ missing or without "
                f"'loss'/'complexity'; skipping ND+val selection."
            )

    return model


def train_symbolic_models(df_X, df_F, df_G, pysr_iters=50, rng_seed=None):
    """
    Train PySR symbolic models for each objective and each constraint.
    Also optionally trains a classifier 'clf_success' if any NaNs appear.

    Uses 80% of available finite data for training and 20% for validation
    inside each _train_single_pysr call (per column), and chooses the
    equation with lowest validation MSE among the nondominated set in
    (train loss, complexity)-space.
    """
    X = df_X.values.astype(float)
    models = {}

    # Objective models
    for col in df_F.columns:
        y = df_F[col].values.astype(float)
        m = _train_single_pysr(X, y, pysr_iters, rng_seed, col, val_fraction=0.2)
        if m is not None:
            models[col] = m

    # Constraint models
    for col in df_G.columns:
        y = df_G[col].values.astype(float)
        m = _train_single_pysr(X, y, pysr_iters, rng_seed, col, val_fraction=0.2)
        if m is not None:
            models[col] = m

    # Classifier model if any NaNs exist
    any_nan_F = np.isnan(df_F.values).any()
    any_nan_G = np.isnan(df_G.values).any() if df_G.shape[1] > 0 else False
    if any_nan_F or any_nan_G:
        row_has_nan_F = np.any(np.isnan(df_F.values), axis=1)
        if df_G.shape[1] > 0:
            row_has_nan_G = np.any(np.isnan(df_G.values), axis=1)
        else:
            row_has_nan_G = np.zeros(df_F.shape[0], dtype=bool)
        row_has_nan = row_has_nan_F | row_has_nan_G
        y_clf = (~row_has_nan).astype(float)

        if np.unique(y_clf).size >= 2:
            print("[train_symbolic_models] Training classifier 'clf_success' for analysis success/failure.")
            m_clf = _train_single_pysr(X, y_clf, pysr_iters, rng_seed,
                                       "clf_success", val_fraction=0.2)
            if m_clf is not None:
                models["clf_success"] = m_clf

    return models


def build_symbolic_surrogates(df_X, df_F, df_G, pysr_iters=50, rng_seed=None):
    """
    Train PySR models once and build shared SymPy expressions and lambdified
    F/G callables to be reused for multiple proposal strategies.

    Returns:
        sympy_exprs : dict of SymPy expressions for f_i, g_j, (and classifier g)
        lambdas_f   : list of callables for objectives
        lambdas_g   : list of callables for constraints (possibly empty)
    or (None, None, None) if nothing could be trained.
    """
    sym_models = train_symbolic_models(df_X, df_F, df_G,
                                       pysr_iters=pysr_iters,
                                       rng_seed=rng_seed)
    if len(sym_models) == 0:
        print("[build_symbolic_surrogates] No symbolic models trained.")
        return None, None, None

    sympy_exprs = get_sympy_exprs_from_pysr_models(sym_models)
    sympy_exprs = add_classifier_constraint(sympy_exprs)

    D = df_X.shape[1]
    x_syms = sp.symbols(f"x0:{D}")

    f_keys = sorted([k for k in sympy_exprs if k.startswith("f")])

    lambdas_f = [sp.lambdify(x_syms, sympy_exprs[k], modules=["numpy"])
                 for k in f_keys]

    # ====================================================
    # CHANGED: Only build lambdas for "physical" constraints
    # (those present in df_G), not for extra classifier-based g's.
    # This keeps the constraint dimension for DSS consistent with df_G.
    # ====================================================
    n_con = df_G.shape[1]
    if n_con > 0:
        physical_g_keys = [f"g{i+1}" for i in range(n_con)]
        lambdas_g = []
        for key in physical_g_keys:
            expr = sympy_exprs.get(key, None)
            if expr is None:
                # No surrogate for this constraint → treat as zero for predictions
                # (no violation). Still enforced in Pyomo via sympy_exprs.
                lambdas_g.append(sp.lambdify(x_syms, 0, modules=["numpy"]))
            else:
                lambdas_g.append(sp.lambdify(x_syms, expr, modules=["numpy"]))
    else:
        lambdas_g = []
    # ====================================================

    if len(lambdas_f) == 0:
        print("[build_symbolic_surrogates] No objective models available.")
        return None, None, None

    return sympy_exprs, lambdas_f, lambdas_g


# ====================================================
# Pyomo model building & solving
# ====================================================

def _extract_solution_random_fallback(pym, D, LB, UB, rng=None):
    """
    Extract solution; if a variable value is None/inf, sample a random backup.
    Additionally, clamp values to [LB[i], UB[i]] to avoid tiny numerical
    violations like x = 1.000001 when the upper bound is 1.0.
    """
    vals = []
    _rng = rng if rng is not None else np.random

    for i in range(D):
        v = pyo.value(pym.x[i], exception=False)
        lb = float(LB[i])
        ub = float(UB[i])

        if v is not None and np.isfinite(v):
            # Clamp to variable bounds to eliminate small overshoots/undershoots
            if v < lb:
                v = lb
            elif v > ub:
                v = ub
            vals.append(float(v))
            continue

        # If solver did not return a valid finite value, fall back to random
        if not np.isfinite(lb):
            lb = -1.0
        if not np.isfinite(ub):
            ub = 1.0
        if ub < lb:
            lb, ub = ub, lb
        if not np.isfinite(lb) and not np.isfinite(ub):
            lb, ub = -1.0, 1.0

        if ub == lb:
            vals.append(lb)
        else:
            try:
                r = _rng.uniform(lb, ub)
            except AttributeError:
                r = np.random.uniform(lb, ub)
            vals.append(float(r))

    # Final rounding to the global ROUND_DECIMALS
    return round_array(vals)


def build_pyomo_from_symbolic(sympy_exprs, LB, UB, D,
                              tchebycheff=False, weights=None, z_star=None,
                              single_obj_idx=None):
    """
    Build a Pyomo model from symbolic expressions.
    One of:
      - single_obj_idx: index of objective to minimize
      - tchebycheff: True for Tchebycheff scalarization with given weights & z_star
    """
    m = pyo.ConcreteModel()
    m.x = pyo.Var(range(D), domain=pyo.Reals,
                  bounds=lambda _m, i: (float(LB[i]), float(UB[i])))

    f_keys = sorted([k for k in sympy_exprs if k.startswith('f')])
    g_keys = sorted([k for k in sympy_exprs if k.startswith('g')])
    pyomo_f = [sympy_to_pyomo(sympy_exprs[k], m) for k in f_keys]
    pyomo_g = [sympy_to_pyomo(sympy_exprs[k], m) for k in g_keys]

    for i, gexpr in enumerate(pyomo_g):
        m.add_component(f'c_{i+1}', pyo.Constraint(expr=gexpr <= 0))

    if single_obj_idx is not None:
        m.obj = pyo.Objective(expr=pyomo_f[single_obj_idx], sense=pyo.minimize)
        return m

    if tchebycheff:
        assert weights is not None and z_star is not None
        m.t = pyo.Var(domain=pyo.Reals, bounds=(0, None))
        m.obj = pyo.Objective(expr=m.t, sense=pyo.minimize)

        w = np.asarray(weights, float).reshape(-1)
        z = np.asarray(z_star, float).reshape(-1)

        for i, fexpr in enumerate(pyomo_f):
            diff = fexpr - float(z[i])
            m.add_component(f't_pos_{i+1}', pyo.Constraint(expr=m.t >= float(w[i]) * diff))
            m.add_component(f't_neg_{i+1}', pyo.Constraint(expr=m.t >= float(w[i]) * (-diff)))
        return m

    raise RuntimeError("Provide single_obj_idx or set tchebycheff=True")


def solve_pyomo_model(pym, scip_exec, feas_tol, time_limit=60, node_limit=10000):
    """
    Run SCIP with feasibility tolerance `feas_tol`.
    """
    s = pyo.SolverFactory('scip', executable=scip_exec)
    s.options['limits/nodes'] = node_limit
    s.options['limits/time'] = time_limit
    s.options['display/verblevel'] = 0
    s.options['numerics/feastol'] = feas_tol
    s.options['numerics/epsilon'] = feas_tol / 10.0
    s.options['lp/disablecutoff'] = 1#True
    s.options['lp/cleanup'] = 1#True
    return s.solve(pym, tee=False)


# ====================================================
# Distance-based subset selection
# ====================================================
def distance_based_subset_selection(df_Ax, df_AF, df_AG,
                                    df_cx, df_cF, df_cG,
                                    feas_tol, eps, eps_dom=0.0,
                                    z_star=None, nadir=None):
    """
    Select one candidate from df_cx based on constrained ε-dominance,
    feasibility, and distance in F-space (via normalized scaling).

    For scaling, if (z_star, nadir) are provided they are used as the
    global ideal/nadir (e.g., [0,0] and [1,1] from global_ideal_nadir).
    Otherwise we fall back to archive-based estimates.
    """
    X_archive = np.asarray(df_Ax.values, float) if len(df_Ax) > 0 else np.zeros((0, 0))
    F_archive = np.asarray(df_AF.values, float) if len(df_AF) > 0 else np.zeros((0, 0))
    G_archive = np.asarray(df_AG.values, float) if len(df_AG) > 0 else np.zeros((0, 0))

    X_cand = np.asarray(df_cx.values, float)
    F_cand = np.asarray(df_cF.values, float)
    G_cand = np.asarray(df_cG.values, float) if df_cG.shape[1] > 0 else np.zeros((len(df_cF), 0))

    m = X_cand.shape[0]
    if m == 0:
        raise ValueError("Candidate set empty")
    if m == 1:
        return X_cand[0].copy()

    n_archive = F_archive.shape[0]

    # --- Archive feasible mask ---
    if n_archive > 0:
        feas_mask_archive = (np.apply_along_axis(is_feasible_row, 1, G_archive, feas_tol)
                             if G_archive.size > 0 else np.ones(n_archive, dtype=bool))
    else:
        feas_mask_archive = np.array([], dtype=bool)

    # --- Build targets_F, targets_G from archive ---
    if n_archive > 0 and np.any(feas_mask_archive):
        F_feas_arch = F_archive[feas_mask_archive]
        F_feas_arch_q = round_to_precision(F_feas_arch)          # quantize
        G_feas_arch = G_archive[feas_mask_archive]
        nd_mask_arch = constrained_nondominated_mask(F_feas_arch_q, G_feas_arch, feas_tol, eps_dom)
        targets_F = F_feas_arch_q[nd_mask_arch]
        targets_G = G_feas_arch[nd_mask_arch]
        target_is_feasible = True
    elif n_archive > 0:
        violations = (np.apply_along_axis(sum_pos_violation, 1, G_archive)
                      if G_archive.size > 0 else np.zeros(n_archive))
        best_idx = int(np.nanargmin(violations))
        f_best_q = round_to_precision(F_archive[[best_idx], :])  # quantize
        targets_F = f_best_q
        targets_G = G_archive[[best_idx], :] if G_archive.size > 0 else np.zeros((1, 0))
        target_is_feasible = False
    else:
        F_cand_q = round_to_precision(F_cand)                    # quantize
        targets_F = np.zeros((0, F_cand_q.shape[1])) if F_cand_q.size > 0 else np.zeros((0, 0))
        targets_G = np.zeros((0, G_cand.shape[1]))
        target_is_feasible = False

    # --- Candidate feasibility mask ---
    cand_feas_mask = (np.apply_along_axis(is_feasible_row, 1, G_cand, feas_tol)
                      if G_cand.size > 0 else np.ones(m, dtype=bool))

    if np.any(cand_feas_mask):
        F_feas_cand = F_cand[cand_feas_mask]
        F_feas_cand_q = round_to_precision(F_feas_cand)          # quantize
        G_feas_cand = G_cand[cand_feas_mask] if G_cand.size > 0 else np.zeros((F_feas_cand.shape[0], 0))
        nd_mask_cand = constrained_nondominated_mask(F_feas_cand_q, G_feas_cand, feas_tol, eps_dom)
        idxs_feas = np.where(cand_feas_mask)[0]
        front0_local = np.where(nd_mask_cand)[0]
        promising_idx = idxs_feas[front0_local].tolist()
    else:
        violations_c = (np.apply_along_axis(sum_pos_violation, 1, G_cand)
                        if G_cand.size > 0 else np.zeros(m))
        best_local = int(np.nanargmin(violations_c))
        promising_idx = [best_local]

    if len(promising_idx) == 1:
        return X_cand[promising_idx[0]].copy()

    X_prom = X_cand[promising_idx]
    F_prom = F_cand[promising_idx]
    F_prom_q = round_to_precision(F_prom)                        # quantize for scaling
    G_prom = G_cand[promising_idx] if G_cand.size > 0 else np.zeros((len(promising_idx), 0))

    # --- Normalization using fixed global ideal/nadir if provided,
    #     otherwise archive/candidate-based estimates.
    if z_star is not None and nadir is not None:
        ideal = np.asarray(z_star, dtype=float).reshape(-1)
        nadir_arr = np.asarray(nadir, dtype=float).reshape(-1)

        n_obj = F_prom_q.shape[1]
        if ideal.size != n_obj:
            if ideal.size > n_obj:
                ideal = ideal[:n_obj]
            else:
                ideal = np.pad(ideal, (0, n_obj - ideal.size), mode='constant')
        if nadir_arr.size != n_obj:
            if nadir_arr.size > n_obj:
                nadir_arr = nadir_arr[:n_obj]
            else:
                nadir_arr = np.pad(nadir_arr, (0, n_obj - nadir_arr.size), mode='constant')
        nadir = nadir_arr
    else:
        if n_archive > 0 and np.any(feas_mask_archive):
            F_arch_for_norm = round_to_precision(F_archive[feas_mask_archive])  # quantize
            ideal = np.nanmin(F_arch_for_norm, axis=0)
            nadir = np.nanmax(F_arch_for_norm, axis=0)
        elif n_archive > 0:
            F_arch_for_norm = round_to_precision(F_archive)                    # quantize
            ideal = np.nanmin(F_arch_for_norm, axis=0)
            nadir = np.nanmax(F_arch_for_norm, axis=0)
        else:
            F_cand_for_norm = round_to_precision(F_cand)                       # quantize
            ideal = np.nanmin(F_cand_for_norm, axis=0)
            nadir = np.nanmax(F_cand_for_norm, axis=0)

    denom = np.maximum(nadir - ideal, eps)

    def normF(Farr):
        return (np.asarray(Farr, float) - ideal) / denom

    if targets_F.shape[0] == 0:
        # no archive targets → just pick the first promising candidate
        return X_prom[0].copy()

    targets_Fn = normF(targets_F)
    F_prom_n = normF(F_prom_q)

    # --- Constrained nondominance on union of targets & promising ---
    union_F = np.vstack([targets_F, F_prom_q])
    union_G = np.vstack([targets_G, G_prom])
    nd_union = constrained_nondominated_mask(union_F, union_G, feas_tol, eps_dom)

    n_targets = targets_F.shape[0]
    prom_union_indices = np.arange(n_targets, n_targets + F_prom.shape[0])

    # elites are those candidate positions that are nondominated in union
    elites_mask = nd_union[prom_union_indices]
    elites = np.where(elites_mask)[0]  # positions inside F_prom

    # --- robust fallback if elites is empty ---
    if elites.size == 0:
        elite_Fn = F_prom_n
        d = euclidean_dist_matrix(elite_Fn, targets_Fn)
        best_rel = int(np.argmax(np.min(d, axis=1)))
        return X_prom[best_rel].copy()

    if elites.size == 1:
        return X_prom[elites[0]].copy()

    elite_Fn = F_prom_n[elites]
    d = euclidean_dist_matrix(elite_Fn, targets_Fn)
    best_rel = int(np.argmax(np.min(d, axis=1)))
    return X_prom[elites[best_rel]].copy()


# ====================================================
# Helpers for G dataframes
# ====================================================

def pad_df_G(df_G, target_cols):
    """Pad constraint DF to target_cols with zeros, or create empty appropriately."""
    if df_G.shape[1] == target_cols:
        return df_G.copy()
    if target_cols == 0:
        return pd.DataFrame(np.zeros((len(df_G), 0)))
    if df_G.shape[1] == 0:
        return pd.DataFrame(
            np.zeros((len(df_G), target_cols)),
            columns=[f"g{i+1}" for i in range(target_cols)],
        )

    pad_w = target_cols - df_G.shape[1]
    zeros = np.zeros((len(df_G), pad_w))
    new = np.hstack([df_G.values, zeros])
    cols = [f"g{i+1}" for i in range(target_cols)]
    return pd.DataFrame(new, columns=cols)


def safe_append_G_row(df_G: pd.DataFrame, g_flat: np.ndarray) -> pd.DataFrame:
    """Safely append constraint vector to df_G, handling zero-column/unconstrained cases."""
    if not isinstance(g_flat, np.ndarray):
        g_flat = np.atleast_1d(g_flat)
    g_flat = g_flat.flatten()

    if df_G.shape[1] == 0:
        return pd.concat([df_G, pd.DataFrame(np.zeros((1, 0)))], ignore_index=True)

    k = df_G.shape[1]
    if g_flat.size == 0:
        row = [0.0] * k
    else:
        row = list(g_flat) + [0.0] * max(0, k - g_flat.size)
        if len(row) > k:
            row = row[:k]

    df_G.loc[len(df_G)] = row
    return df_G


# ====================================================
# Warm-up: per concept feasible-by-singleobj
# ====================================================

def find_feasible_by_singleobj(obj_idx, df_X, df_F, df_G, evaluate_function,
                               LB, UB, D, scip_exec,
                               feas_tol,
                               max_attempts=50, eps=1e-6,
                               pysr_iters=60, rng_seed=None):
    """
    Minimize objective f_{obj_idx} with surrogate models until a feasible
    (or best effort) point is found or max_attempts reached.
    """
    attempt = 0
    while attempt < max_attempts:
        attempt += 1

        sym_models = train_symbolic_models(df_X, df_F, df_G,
                                           pysr_iters=pysr_iters,
                                           rng_seed=rng_seed)
        sympy_exprs = get_sympy_exprs_from_pysr_models(sym_models)
        sympy_exprs = add_classifier_constraint(sympy_exprs)

        if len(sympy_exprs) == 0:
            print(f"[Min f{obj_idx+1}] No symbolic models to build Pyomo model; aborting warm-up.")
            break

        pym = build_pyomo_from_symbolic(sympy_exprs, LB, UB, D,
                                        single_obj_idx=obj_idx)

        # robust solver call
        try:
            results = solve_pyomo_model(pym, scip_exec, feas_tol)
        except ApplicationError as e:
            print(f"[Min f{obj_idx+1}] solver crashed (ApplicationError): {e}; retrain.")
            continue
        except Exception as e:
            print(f"[Min f{obj_idx+1}] solver crashed (unexpected error): {e}; retrain.")
            continue

        if (results.solver.status != pyo.SolverStatus.ok or
                results.solver.termination_condition != pyo.TerminationCondition.optimal):
            print(f"[Min f{obj_idx+1}] solver non-optimal (attempt {attempt}); retrain.")
            continue

        rng_local = np.random.default_rng(rng_seed) if rng_seed is not None else None
        x_new = _extract_solution_random_fallback(pym, D, LB, UB, rng=rng_local)
        dists = normalized_distance_to_archive(x_new, df_X, LB, UB, eps)
        if dists.size > 0 and np.any(dists < eps):
            print(f"[Min f{obj_idx+1}] Too close to archive (attempt {attempt}); retry.")
            continue

        n_obj = df_F.shape[1]
        n_con = df_G.shape[1]
        F_new, G_new, success = safe_true_evaluate(evaluate_function, x_new, n_obj, n_con)

        df_X.loc[len(df_X)] = round_array(x_new)
        df_F.loc[len(df_F)] = F_new.flatten()
        df_G_updated = safe_append_G_row(df_G, np.atleast_1d(G_new).flatten())

        print(f"[Min f{obj_idx+1}] #{attempt}: X={round_array(x_new)}, "
              f"F={round_array(F_new.flatten())}, "
              f"G={round_array(np.atleast_1d(G_new).flatten())}")

        if success and (G_new.size == 0 or
                        np.all(np.atleast_1d(G_new).flatten() <= feas_tol)):
            print(f"[Min f{obj_idx+1}] feasible.")
            return x_new, F_new.flatten(), np.atleast_1d(G_new).flatten(), df_G_updated

        print(f"[Min f{obj_idx+1}] infeasible or failed analysis; retrain.")
        df_G = df_G_updated

    print(f"[Min f{obj_idx+1}] max attempts reached without feasible.")
    return None


# ====================================================
# Tchebycheff: per-concept proposal (no true eval)
# ====================================================

def _build_scaled_sympy_for_tchebycheff(sympy_exprs, z_star, nadir):
    """
    Scale objectives for Tchebycheff scalarization.

    Both z_star and nadir are first quantized to 1e-6 to be consistent
    with how ideal/nadir are computed globally.
    """
    f_keys = sorted([k for k in sympy_exprs if k.startswith('f')])
    g_keys = sorted([k for k in sympy_exprs if k.startswith('g')])
    n_f = len(f_keys)

    z_star_q = round_to_precision(z_star)   # quantize
    nadir_q = round_to_precision(nadir)     # quantize

    scaled_sympy = {}
    for i, k in enumerate(f_keys):
        denom = float(nadir_q[i] - z_star_q[i]) if abs(nadir_q[i] - z_star_q[i]) > 1e-12 else 1.0
        scaled_sympy[k] = (sympy_exprs[k] - float(z_star_q[i])) / denom
    for k in g_keys:
        scaled_sympy[k] = sympy_exprs[k]
    return scaled_sympy, f_keys, g_keys, n_f, len(g_keys)


def _build_z_targets(n_f, n_points):
    """Build a set of target zs along a line in objective space."""
    v_parallel = np.array([-1.0, 1.0]) / np.sqrt(2.0)
    s_vals = np.linspace(-1.0/np.sqrt(2.0), 1.0/np.sqrt(2.0), n_points)
    z_targets = []
    for s in s_vals:
        z = np.zeros(n_f)
        z[0:2] = s * v_parallel if n_f >= 2 else np.array([s])
        z_targets.append(z)
    return np.vstack(z_targets)


def _filter_candidates_far_from_archive(cand_X, df_X, LB, UB, eps):
    """Keep only candidates that are at least eps away from all archive points."""
    def is_far(x_row):
        if len(df_X) == 0:
            return True
        d = normalized_distance_to_archive(x_row, df_X, LB, UB, eps)
        return d.size == 0 or np.all(d >= eps)

    return np.array([x for x in cand_X if is_far(x)])


def _predict_symbolic_batch(eligible_X, lambdas_f, lambdas_g):
    k = eligible_X.shape[0]
    n_f = len(lambdas_f)
    n_g = len(lambdas_g) if lambdas_g is not None else 0
    Fp = np.zeros((k, n_f))
    Gp = np.zeros((k, n_g)) if n_g > 0 else np.zeros((k, 0))
    for i in range(k):
        tup = tuple(eligible_X[i].tolist())
        for j, lf in enumerate(lambdas_f):
            try:
                Fp[i, j] = float(lf(*tup))
            except Exception:
                Fp[i, j] = np.nan
        if n_g > 0:
            for j, lg in enumerate(lambdas_g):
                try:
                    Gp[i, j] = float(lg(*tup))
                except Exception:
                    Gp[i, j] = np.nan
    return Fp, Gp

def _solve_one_z_target(idx, zt,
                               scaled_sympy, LB, UB, D,
                               w_default, scip_exec, feas_tol,
                               rng_seed, eps):
    """
    Worker process for a single z_target.
    Returns x_try (numpy array) or None.
    """
    try:
        pym = build_pyomo_from_symbolic(
            scaled_sympy, LB, UB, D,
            tchebycheff=True, weights=w_default, z_star=zt
        )
        # small regularization term
        if hasattr(pym, "obj"):
            pym.obj.expr += 1e-6 * sum(pym.x[i]**2 for i in range(D))
    except Exception as e:
        print(f"[TCH-W{idx}] build failed: {e}")
        return None

    try:
        solve_pyomo_model(pym, scip_exec, feas_tol)
    except Exception as e:
        print(f"[TCH-W{idx}] solver crash: {e}")
        return None

    # extract solution (with RNG for tie-breaking)
    try:
        if rng_seed is not None:
            rng_local = np.random.default_rng(rng_seed + idx)
        else:
            rng_local = None

        x_try = _extract_solution_random_fallback(pym, D, LB, UB, rng=rng_local)
        return x_try
    except Exception as e:
        print(f"[TCH-W{idx}] extraction failed: {e}")
        return None


def tchebycheff_propose_one(df_X, LB, UB, D,
                            z_star, nadir, n_points,
                            df_AF_global, df_AG_global,
                            scip_exec, feas_tol,
                            eps, eps_dom,
                            sympy_exprs, lambdas_f, lambdas_g,
                            rng_seed=None,
                            n_workers=-1):
    """
    Full version with joblib parallelization of z-target solves.
    n_workers = -1 → use all cores
    """
    print("[TCH] >>> Starting tchebycheff_propose_one (parallel via joblib)")

    # ----------------------------------------------------
    # Preconditions
    # ----------------------------------------------------
    if sympy_exprs is None or lambdas_f is None or len(lambdas_f) == 0:
        print("[TCH] No symbolic models; returning None.")
        return None, None, None

    scaled_sympy, f_keys, g_keys, n_f, n_g = _build_scaled_sympy_for_tchebycheff(
        sympy_exprs, z_star, nadir
    )
    print(f"[TCH] f_keys={f_keys}, g_keys={g_keys}, n_f={n_f}, n_g={n_g}")

    if n_f == 0:
        print("[TCH] No objective models; returning None.")
        return None, None, None

    # ----------------------------------------------------
    # Build z_targets
    # ----------------------------------------------------
    print("[TCH] Building z_targets...")
    z_targets = _build_z_targets(n_f, n_points)
    print(f"[TCH] {len(z_targets)} z_targets created.")

    w_default = np.ones(n_f) / float(max(1, n_f))

    # ----------------------------------------------------
    # PARALLEL solve for each z_target
    # ----------------------------------------------------
    print("[TCH] Launching parallel Pyomo solves (joblib)...")

    results = Parallel(n_jobs=n_workers)(
        delayed(_solve_one_z_target)(
            idx, zt,
            scaled_sympy, LB, UB, D,
            w_default, scip_exec, feas_tol,
            rng_seed, eps
        )
        for idx, zt in enumerate(z_targets)
    )

    # ----------------------------------------------------
    # Gather & deduplicate
    # ----------------------------------------------------
    cand_X = []
    for x_try in results:
        if x_try is not None:
            if not any(np.allclose(x_try, xx, atol=eps) for xx in cand_X):
                cand_X.append(x_try)
                print(f"[TCH] Candidate added: {round_array(x_try)}")

    print(f"[TCH] Total candidates collected: {len(cand_X)}")

    if len(cand_X) == 0:
        print("[TCH] No candidates — returning None.")
        return None, None, None

    cand_X = np.vstack(cand_X)

    # ----------------------------------------------------
    # Candidate filtering
    # ----------------------------------------------------
    print("[TCH] Filtering candidates vs. archive...")
    eligible_X = _filter_candidates_far_from_archive(cand_X, df_X, LB, UB, eps)
    print(f"[TCH] Eligible candidates: {len(eligible_X)}")

    if eligible_X.size == 0:
        print("[TCH] All candidates too close; returning None.")
        return None, None, None

    # ----------------------------------------------------
    # Predict symbolic outputs
    # ----------------------------------------------------
    F_pred, G_pred = _predict_symbolic_batch(eligible_X, lambdas_f, lambdas_g)

    df_cx = pd.DataFrame(eligible_X)
    df_cF = pd.DataFrame(F_pred, columns=[f"f{i+1}" for i in range(F_pred.shape[1])])

    # ====================================================
    # CHANGED: Align candidate constraint dimension with GLOBAL archive G
    # so that distance_based_subset_selection can safely stack them.
    # ====================================================
    max_g_global = df_AG_global.shape[1]  # number of constraint columns globally

    if max_g_global == 0:
        df_cG = pd.DataFrame(np.zeros((len(eligible_X), 0)))
    else:
        G_adj = G_pred
        if G_adj.shape[1] > max_g_global:
            # Truncate extra surrogate constraints (e.g. classifier-based)
            G_adj = G_adj[:, :max_g_global]
        elif G_adj.shape[1] < max_g_global:
            # Pad missing constraints with zeros (no violation)
            pad = max_g_global - G_adj.shape[1]
            G_adj = np.hstack([G_adj, np.zeros((G_adj.shape[0], pad))])

        df_cG = pd.DataFrame(
            G_adj,
            columns=[f"g{i+1}" for i in range(max_g_global)],
        )

    # dummy archive sizing (DSS requirement)
    if len(df_AF_global) > 0:
        dummy_Ax = pd.DataFrame(np.zeros((len(df_AF_global), 1)))
    else:
        dummy_Ax = pd.DataFrame(np.zeros((0, 1)))

    # ----------------------------------------------------
    # DSS selection
    # ----------------------------------------------------
    chosen_row = distance_based_subset_selection(
        dummy_Ax, df_AF_global, df_AG_global,
        df_cx, df_cF, df_cG,
        feas_tol=feas_tol, eps=eps, eps_dom=eps_dom,
        z_star=z_star, nadir=nadir,
    )

    idx = None
    for i in range(len(eligible_X)):
        if np.allclose(chosen_row, eligible_X[i], atol=1e-12):
            idx = i
            break
    if idx is None:
        idx = 0

    return eligible_X[idx], F_pred[idx], (G_pred[idx] if G_pred.shape[1] > 0 else np.zeros((0,)))


def singleobj_propose_one(obj_idx, df_X,
                          LB, UB, D,
                          scip_exec, feas_tol,
                          eps,
                          sympy_exprs, lambdas_f, lambdas_g,
                          max_attempts=5, rng_seed=None):
    """
    Surrogate-based single-objective proposal (no true evaluation):
    - uses pre-trained symbolic models (sympy_exprs, lambdas_f/g),
    - solves a MINLP to minimize f_{obj_idx},
    - returns (x, F_pred, G_pred) based on symbolic models.

    Robust to numerical issues in the lambdified expressions (e.g., division by zero):
    prediction errors are caught and replaced by NaN instead of raising.
    """
    if sympy_exprs is None or lambdas_f is None or len(lambdas_f) == 0:
        print(f"[SingleObj f{obj_idx+1}] No symbolic models; aborting.")
        return None, None, None

    attempt = 0
    while attempt < max_attempts:
        attempt += 1

        # Build and solve Pyomo single-objective model
        try:
            pym = build_pyomo_from_symbolic(
                sympy_exprs, LB, UB, D,
                single_obj_idx=obj_idx
            )
        except Exception as e:
            print(f"[SingleObj f{obj_idx+1}] build failed: {e}")
            return None, None, None

        try:
            results = solve_pyomo_model(pym, scip_exec, feas_tol)
        except ApplicationError as e:
            print(f"[SingleObj f{obj_idx+1}] solver crash (ApplicationError): {e}; retry.")
            continue
        except Exception as e:
            print(f"[SingleObj f{obj_idx+1}] solver crash (unexpected): {e}; retry.")
            continue

        if (results.solver.status != pyo.SolverStatus.ok or
                results.solver.termination_condition != pyo.TerminationCondition.optimal):
            print(f"[SingleObj f{obj_idx+1}] solver non-optimal (attempt {attempt}); retry.")
            continue

        rng_local = np.random.default_rng(rng_seed) if rng_seed is not None else None
        x_new = _extract_solution_random_fallback(pym, D, LB, UB, rng=rng_local)

        # Avoid points too close to archive
        dists = normalized_distance_to_archive(x_new, df_X, LB, UB, eps)
        if dists.size > 0 and np.any(dists < eps):
            print(f"[SingleObj f{obj_idx+1}] Too close to archive (attempt {attempt}); retry.")
            continue

        # Predict F,G using shared lambdas, guarding against numerical errors
        tup = tuple(x_new.tolist())
        F_vals = []
        for lf in lambdas_f:
            try:
                F_vals.append(float(lf(*tup)))
            except Exception:
                F_vals.append(np.nan)
        F_pred = np.array(F_vals, dtype=float)

        if lambdas_g is not None and len(lambdas_g) > 0:
            G_vals = []
            for lg in lambdas_g:
                try:
                    G_vals.append(float(lg(*tup)))
                except Exception:
                    G_vals.append(np.nan)
            G_pred = np.array(G_vals, dtype=float)
        else:
            G_pred = np.zeros((0,), dtype=float)

        print(f"[SingleObj f{obj_idx+1}] Proposal x={round_array(x_new)}, "
              f"F_pred={round_array(F_pred)}, "
              f"G_pred={round_array(G_pred)}")

        return x_new, F_pred, G_pred

    print(f"[SingleObj f{obj_idx+1}] max attempts reached without proposal.")
    return None, None, None


# ====================================================
# Global ideal/nadir and Pareto plotting
# ====================================================
def global_ideal_nadir(archives_dict, feas_tol, eps_dom=0.0, alpha=0.0):
    """
    Compute global ideal and nadir across all concepts.

    - Take ALL feasible solutions from the archive (across all concepts).
    - Quantize F-values to 1e-6.
    - Perform nondominated sorting in F-space using the quantized F.
    - On the resulting nondominated set, define:
        ideal = component-wise min over quantized F
        nadir = component-wise max over quantized F

    Fallbacks for <= 1 feasible point also use quantized F.

    Parameters
    ----------
    archives_dict : dict
        Mapping concept -> archive dict with keys "df_F" and "df_G".
    feas_tol : float
        Feasibility tolerance for constraints (g <= feas_tol considered feasible).
    eps_dom : float, optional
        Currently unused here but kept for interface compatibility.
    alpha : float, optional
        Currently unused here but kept for backward compatibility.

    Returns
    -------
    ideal : np.ndarray
        Global ideal point in objective space.
    nadir : np.ndarray
        Global nadir point in objective space.
    """
    F_list, G_list = [], []

    # Collect all F and G across concepts
    for _, d in archives_dict.items():
        Fvals = np.asarray(d["df_F"].values, float)
        if d["df_G"].shape[1] > 0:
            Gvals = np.asarray(d["df_G"].values, float)
        else:
            Gvals = np.zeros((len(Fvals), 0), dtype=float)

        if Fvals.size == 0:
            continue

        F_list.append(Fvals)
        G_list.append(Gvals)

    if len(F_list) == 0:
        raise ValueError("global_ideal_nadir: archives_dict contains no F values.")

    F_all = np.vstack(F_list)
    F_all_q = round_to_precision(F_all)  # quantize F for ideal/nadir & sorting
    G_all = np.vstack(G_list)

    # Finite objectives based on quantized F
    finite_F = np.all(np.isfinite(F_all_q), axis=1)

    # Feasibility via feas_tol
    if G_all.shape[1] > 0:
        G_clean = np.where(np.isnan(G_all), np.inf, G_all)
        feas_G = np.all(G_clean <= feas_tol, axis=1)
    else:
        feas_G = np.ones(F_all.shape[0], dtype=bool)

    feas_mask = finite_F & feas_G
    n_feas = int(np.sum(feas_mask))

    if n_feas >= 2:
        # Use nondominated feasible points to define ideal/nadir
        F_feas_q = F_all_q[feas_mask]
        nd_mask_F = is_nondominated(F_feas_q)
        F_pareto_q = F_feas_q[nd_mask_F]
        ideal = np.min(F_pareto_q, axis=0)
        nadir = np.max(F_pareto_q, axis=0)
        ideal = round_to_precision(ideal)
        nadir = round_to_precision(nadir)
    elif n_feas == 1:
        # Single feasible point: build a simple box around it
        f_q = F_all_q[feas_mask][0]
        ideal = 0.5 * f_q
        nadir = 1.5 * f_q
        ideal = round_to_precision(ideal)
        nadir = round_to_precision(nadir)
    else:
        # No feasible points: pick best (least violating) as a center
        if G_all.shape[1] > 0:
            G_violation = np.where(np.isnan(G_all), 1e6, G_all)
            violations = np.sum(np.maximum(G_violation, 0.0), axis=1)
            best_idx = int(np.nanargmin(violations))
            f_best_q = F_all_q[best_idx]
        else:
            F_for_score = np.where(np.isnan(F_all_q), np.inf, F_all_q)
            scores = np.sum(F_for_score, axis=1)
            best_idx = int(np.nanargmin(scores))
            f_best_q = F_all_q[best_idx]

        ideal = 0.5 * f_best_q
        nadir = 1.5 * f_best_q
        ideal = round_to_precision(ideal)
        nadir = round_to_precision(nadir)

    return ideal, nadir


def plot_and_save_global_pareto(outdir: Path, archives_dict,
                                feas_tol, eps_dom=0.0, show=True):
    Ffeas_list = []
    Gfeas_list = []
    for _, d in archives_dict.items():
        Fv = d["df_F"].values
        Gv = d["df_G"].values if d["df_G"].shape[1] > 0 \
            else np.zeros((len(d["df_F"]), 0))
        feas_G = (np.all(np.where(np.isnan(Gv), np.inf, Gv) <= feas_tol, axis=1)
                  if Gv.size > 0 else np.ones(len(Fv), bool))
        feas_F = np.all(np.isfinite(Fv), axis=1)
        mask = feas_G & feas_F
        Ffeas_list.append(Fv[mask])
        Gfeas_list.append(Gv[mask])

    if len(Ffeas_list) == 0:
        print("No feasible points to plot.")
        return

    Ffeas = np.vstack(Ffeas_list)
    Gfeas = np.vstack(Gfeas_list)
    nd = constrained_nondominated_mask(Ffeas, Gfeas, feas_tol, eps_dom)

    fig = plt.figure(figsize=(6, 6))
    plt.scatter(Ffeas[:, 0], Ffeas[:, 1],
                facecolors='none', edgecolors='green',
                s=60, label='Feasible')
    plt.scatter(Ffeas[nd, 0], Ffeas[nd, 1], s=60, label='Constrained ε-Pareto')
    plt.xlabel('f1')
    plt.ylabel('f2')
    plt.title('Constrained ε-Nondominated Solutions (Global)')
    plt.legend()
    plt.grid(True)
    outpath = outdir / "pareto_global.png"
    fig.savefig(outpath, dpi=150, bbox_inches="tight")
    if show:
        plt.show()
    plt.close(fig)


# ====================================================
# Experiment orchestration
# ====================================================

def build_global_FG(archives):
    """Concatenate F/G across all concepts, padding G where needed."""
    allF, allG = [], []
    max_g = max(A["df_G"].shape[1] for A in archives.values())
    for _, A in archives.items():
        allF.append(A["df_F"])
        allG.append(pad_df_G(A["df_G"], max_g))
    return (
        pd.concat(allF, axis=0, ignore_index=True),
        pd.concat(allG, axis=0, ignore_index=True),
    )


def initialize_archives(problem, lhs_mult, n_obj):
    """
    Perform initial LHS sampling and true evaluation for each concept.
    Returns dict of archives keyed by concept name.
    """
    archives = {}
    for sec, cfg in problem["concepts"].items():
        D = int(cfg["D"])
        LB = np.array(cfg["xl"], float)
        UB = np.array(cfg["xu"], float)
        n0 = lhs_mult * D
        X0 = sample_lhs_qmc(n0, D, LB, UB)
        X0 = round_array(X0)

        # infer constraint dimension
        n_con = None
        for xi in X0:
            try:
                _, G_tmp = cfg["eval_func"](xi)
                G_tmp = np.atleast_2d(G_tmp)
                n_con = G_tmp.shape[1]
                break
            except Exception:
                continue
        if n_con is None:
            n_con = 0
            print(f"[Init] Warning: could not infer constraint dimension for {sec}; assuming unconstrained.")

        F0 = np.zeros((n0, n_obj))
        G0 = np.zeros((n0, n_con)) if n_con > 0 else np.zeros((n0, 0))

        for i in range(len(X0)):
            F_i, G_i, success = safe_true_evaluate(cfg["eval_func"], X0[i], n_obj, n_con)
            F0[i, :] = F_i
            if n_con > 0:
                G0[i, :] = G_i
            if not success:
                print(f"[Init] {sec}: LHS sample {i} analysis failed, storing NaNs in F/G.")

        df_X = pd.DataFrame(X0, columns=[f"x{i}" for i in range(D)])
        df_F = pd.DataFrame(F0, columns=[f"f{i+1}" for i in range(n_obj)])
        if n_con > 0:
            df_G = pd.DataFrame(G0, columns=[f"g{i+1}" for i in range(n_con)])
        else:
            df_G = pd.DataFrame(np.zeros((len(X0), 0)))

        archives[sec] = {
            "D": D,
            "LB": LB,
            "UB": UB,
            "df_X": df_X,
            "df_F": df_F,
            "df_G": df_G,
            "eval_func": cfg["eval_func"],
        }
        print(f"[Init] {sec}: {len(df_X)} LHS; constraints={df_G.shape[1]}")

    return archives


def warmup_archives(archives, scip_exec, feas_tol,
                    max_attempts_singleobj, eps, pysr_iters, seed):
    """Per-concept warm-up: minimize f1 then f2 until feasible."""
    for sec, A in archives.items():
        print(f"\n[Warm-up] Concept: {sec} → minimize f1 until feasible")
        ret = find_feasible_by_singleobj(
            0, A["df_X"], A["df_F"], A["df_G"], A["eval_func"],
            A["LB"], A["UB"], A["D"], scip_exec=scip_exec,
            feas_tol=feas_tol,
            max_attempts=max_attempts_singleobj,
            eps=eps,
            pysr_iters=pysr_iters,
            rng_seed=seed,
        )
        if ret is not None:
            _, _, _, A["df_G"] = ret

        print(f"[Warm-up] Concept: {sec} → minimize f2 until feasible")
        ret = find_feasible_by_singleobj(
            1, A["df_X"], A["df_F"], A["df_G"], A["eval_func"],
            A["LB"], A["UB"], A["D"], scip_exec=scip_exec,
            feas_tol=feas_tol,
            max_attempts=max_attempts_singleobj,
            eps=eps,
            pysr_iters=pysr_iters,
            rng_seed=seed,
        )
        if ret is not None:
            _, _, _, A["df_G"] = ret


def apply_random_fallback(archives, no_max_samples, scip_exec,
                          feas_tol, eps, eps_dom, seed, alpha):
    """
    Random fallback when Tchebycheff returns no proposals.
    Returns updated (archives, df_AF_global, df_AG_global, z_star, nadir).
    """
    remaining = no_max_samples - sum(len(A["df_X"]) for A in archives.values())
    if remaining <= 0:
        print("No proposals from any concept and budget exhausted; stopping loop.")
        return archives, None, None, None, None

    print("No proposals from any concept; using random fallback sampling.")

    rand_sec = random.choice(list(archives.keys()))
    A_rand = archives[rand_sec]

    x_rand = sample_lhs_qmc(
        n_samples=1,
        dim=A_rand["D"],
        lb=A_rand["LB"],
        ub=A_rand["UB"],
    )[0]
    x_rand = round_array(x_rand)
    n_obj = A_rand["df_F"].shape[1]
    n_con = A_rand["df_G"].shape[1]
    F_rand, G_rand, success = safe_true_evaluate(
        A_rand["eval_func"], x_rand, n_obj, n_con
    )

    A_rand["df_X"].loc[len(A_rand["df_X"])] = round_array(x_rand)
    A_rand["df_F"].loc[len(A_rand["df_F"])] = F_rand.flatten()
    A_rand["df_G"] = safe_append_G_row(
        A_rand["df_G"],
        np.atleast_1d(G_rand).flatten(),
    )

    print(f"[Random Fallback] Concept={rand_sec}, "
          f"x={round_array(x_rand)}, "
          f"F_true={round_array(F_rand.flatten())}, "
          f"G_true={round_array(np.atleast_1d(G_rand).flatten())}, "
          f"success={success}")

    df_AF_global, df_AG_global = build_global_FG(archives)
    z_star, nadir = global_ideal_nadir(archives, feas_tol, eps_dom, alpha)
    print(f"[Global] z*={round_array(z_star)}, nadir={round_array(nadir)}")

    return archives, df_AF_global, df_AG_global, z_star, nadir


def select_and_evaluate_proposal(proposals, archives,
                                 df_AF_global, df_AG_global,
                                 scip_exec, feas_tol, eps, eps_dom, seed, alpha,
                                 z_star, nadir):
    """
    From per-concept proposals (in surrogate space),
    choose one globally using DSS and evaluate true model.

    Normalization inside DSS uses the supplied global (z_star, nadir),
    typically [0,0] and [1,1] from global_ideal_nadir.
    """
    idx_to_concept = {i: p["concept"] for i, p in enumerate(proposals)}
    df_cx_union = pd.DataFrame(
        np.arange(len(proposals)).reshape(-1, 1), columns=["concept_idx"]
    )
    df_cF_union = pd.DataFrame(
        np.vstack([p["F_pred"] for p in proposals]),
        columns=["f1", "f2"],
    )

    max_g = df_AG_global.shape[1]
    G_union = []
    for p in proposals:
        g = np.atleast_1d(p["G_pred"]).astype(float)
        if max_g == 0:
            G_union.append(np.zeros((1, 0)))
        else:
            if g.shape[0] > max_g:
                g = g[:max_g]
            pad = max_g - g.shape[0]
            if pad > 0:
                g = np.hstack([g, np.zeros(pad)])
            G_union.append(g.reshape(1, -1))

    df_cG_union = pd.DataFrame(
        np.vstack(G_union),
        columns=[f"g{i+1}" for i in range(max_g)] if max_g > 0 else [],
    )

    dummy_Ax = pd.DataFrame(np.zeros((len(df_AF_global), 1))) \
        if len(df_AF_global) > 0 else pd.DataFrame(np.zeros((0, 1)))

    chosen_marker = distance_based_subset_selection(
        dummy_Ax, df_AF_global, df_AG_global,
        df_cx_union, df_cF_union, df_cG_union,
        feas_tol=feas_tol, eps=eps, eps_dom=eps_dom,
        z_star=z_star, nadir=nadir,
    )
    chosen_idx = int(np.round(chosen_marker[0])) \
        if chosen_marker.ndim == 1 else int(np.round(chosen_marker[0, 0]))
    chosen_sec = idx_to_concept.get(chosen_idx, proposals[0]["concept"])
    chosen = next(p for p in proposals if p["concept"] == chosen_sec)
    print(f"[Global DSS] Selected concept = {chosen_sec}")

    A = archives[chosen_sec]
    n_obj = A["df_F"].shape[1]
    n_con = A["df_G"].shape[1]
    F_true, G_true, success = safe_true_evaluate(
        A["eval_func"], chosen["x"], n_obj, n_con
    )

    A["df_X"].loc[len(A["df_X"])] = round_array(chosen["x"])
    A["df_F"].loc[len(A["df_F"])] = F_true.flatten()

    g_true = np.atleast_1d(G_true).flatten()
    if A["df_G"].shape[1] == 0 and n_con > 0:
        A["df_G"] = pd.DataFrame(
            np.zeros((len(A["df_F"]) - 1, n_con)),
            columns=[f"g{i+1}" for i in range(n_con)],
        )
    if A["df_G"].shape[1] < n_con:
        pad_cols = n_con - A["df_G"].shape[1]
        for k in range(pad_cols):
            A["df_G"][f"g{A['df_G'].shape[1] + k + 1}"] = 0.0

    A["df_G"] = safe_append_G_row(A["df_G"], g_true)

    print(f"[True] {chosen_sec}: success={success}, "
          f"F_true={round_array(F_true.flatten())}, "
          f"G_true={round_array(g_true)}")

    df_AF_global, df_AG_global = build_global_FG(archives)
    z_new, n_new = global_ideal_nadir(archives, feas_tol, eps_dom, alpha)
    print(f"[Global] z*={round_array(z_new)}, nadir={round_array(n_new)}")

    return archives, df_AF_global, df_AG_global, z_new, n_new


def archives_snapshot(archives):
    """Lightweight snapshot suitable for pickling."""
    snap = {}
    for sec, A in archives.items():
        snap[sec] = {
            "D": int(A["D"]),
            "LB": A["LB"].tolist(),
            "UB": A["UB"].tolist(),
            "df_X": A["df_X"],
            "df_F": A["df_F"],
            "df_G": A["df_G"],
        }
    return snap


def run_experiment(CONFIG):
    problem = CONFIG["problem"]

    # Problem name
    try:
        problem_key = [k for k, v in globals().items() if v is problem][0]
    except IndexError:
        problem_key = "unknown_problem"
    base_name = problem_key.replace("problem_", "")

    # Suffix based on #concepts
    try:
        num_concepts = len(problem["concepts"])
        suffix = "multi" if num_concepts > 1 else "single"
    except Exception:
        num_concepts = 1
        suffix = "single"

    run_folder_name = f"{base_name}_{suffix}"

    scip_exec = CONFIG["scip_exec"]
    seeds = CONFIG["seeds"]
    eps = CONFIG["eps"]
    feas_tol = CONFIG["feas_tol"]
    eps_dom = CONFIG.get("eps_dom", 0.0)
    alpha = CONFIG.get("alpha", 0.0)
    max_attempts_singleobj = CONFIG["max_attempts_singleobj"]
    max_total_weights = CONFIG["max_total_weights"]
    pysr_iters = CONFIG["pysr_iters"]
    results_root = Path(CONFIG["results_root"])
    plot_flag = CONFIG["plot"]

    concepts = problem["concepts"]
    D_per_concept = {sec: int(cfg["D"]) for sec, cfg in concepts.items()}
    D_total = sum(D_per_concept.values())

    lhs_per_dim = CONFIG.get("lhs_per_dim", 10)
    budget_per_dim = CONFIG.get("budget_per_dim", 50)

    lhs_mult = lhs_per_dim
    default_no_max_samples = budget_per_dim * D_total
    no_max_samples = int(CONFIG.get("no_max_samples", default_no_max_samples))

    print(f"[Config] lhs_per_dim={lhs_per_dim} → n0 = lhs_per_dim·D per concept.")
    print(f"[Config] budget_per_dim={budget_per_dim}, D_total={D_total} → "
          f"no_max_samples={no_max_samples} total evals across all concepts.")

    for seed in seeds:
        np.random.seed(seed)
        random.seed(seed)
        print("\n" + "=" * 70)
        print(f"Multi-Objective Multi-Concept (seed={seed})")
        print("=" * 70)

        run_dir = results_root / run_folder_name / f"Seed-{seed}"
        run_dir.mkdir(parents=True, exist_ok=True)
        (run_dir / "config.json").write_text(json.dumps(CONFIG, indent=2, default=str))

        # 1) Initialization
        n_obj = 2
        archives = initialize_archives(problem, lhs_mult, n_obj)

        # 2) Warm-up
        warmup_archives(archives, scip_exec, feas_tol,
                        max_attempts_singleobj, eps, pysr_iters, seed)

        # 3) Global ideal / nadir and global archive
        z_star, nadir = global_ideal_nadir(archives, feas_tol, eps_dom, alpha)
        print(f"\n[Global] Initial ideal: {round_array(z_star)}, "
              f"nadir: {round_array(nadir)}")

        df_AF_global, df_AG_global = build_global_FG(archives)

        current_total = sum(len(A["df_X"]) for A in archives.values())
        print(f"\n[Loop] Current archive size after init+warmup: {current_total}")
        print(f"[Loop] Total eval budget (no_max_samples): {no_max_samples}")
        
        if current_total >= no_max_samples:
            print("[Loop] Budget already exhausted by initial + warm-up evaluations; skipping main loop.")
        else:
            try:
                iter_count = 0
                while sum(len(A["df_X"]) for A in archives.values()) < no_max_samples:
                    iter_count += 1
                    current_total = sum(len(A["df_X"]) for A in archives.values())
                    print(f"\n=== Global Iteration {iter_count} ===")
                    print("Archive sizes:",
                        {sec: len(A["df_X"]) for sec, A in archives.items()})
                    print(f"Total evals so far: {current_total}/{no_max_samples}")

                    # Flag: do extra single-objective (f1,f2) proposals only once
                    # every max_total_weights iterations (e.g. 31, 62, 93, ...).
                    do_extra_single_obj = (iter_count % max_total_weights == 0)
                    if do_extra_single_obj:
                        print(f"[Loop] Extra single-objective proposals enabled "
                            f"(iter {iter_count}, multiple of max_total_weights={max_total_weights}).")
                    else:
                        print(f"[Loop] Skipping extra single-objective proposals this iteration.")

                    # 3a) Per-concept proposals:
                    #     - Tchebycheff along max_total_weights directions
                    #     - plus two extra single-objective searches (min f1, min f2)
                    #       only when do_extra_single_obj is True.
                    proposals = []
                    for sec, A in archives.items():
                        print(f"\n[Loop] Concept: {sec} → build surrogates")
                        sympy_exprs, lambdas_f, lambdas_g = build_symbolic_surrogates(
                            A["df_X"], A["df_F"], A["df_G"],
                            pysr_iters=pysr_iters,
                            rng_seed=seed,
                        )
                        if sympy_exprs is None:
                            print(f"[{sec}] Skipping proposals: no surrogates available.")
                            continue

                        # (1) Tchebycheff proposals along weight line
                        x_cand, Fp, Gp = tchebycheff_propose_one(
                            A["df_X"],
                            A["LB"], A["UB"], A["D"],
                            z_star, nadir, n_points=max_total_weights,
                            df_AF_global=df_AF_global, df_AG_global=df_AG_global,
                            scip_exec=scip_exec,
                            feas_tol=feas_tol,
                            eps=eps,
                            eps_dom=eps_dom,
                            sympy_exprs=sympy_exprs,
                            lambdas_f=lambdas_f,
                            lambdas_g=lambdas_g,
                            rng_seed=seed,
                        )
                        if x_cand is not None:
                            proposals.append({
                                "concept": sec,
                                "x": x_cand,
                                "F_pred": Fp,
                                "G_pred": (Gp if Gp is not None else np.zeros((0,))),
                            })
                            print(f"[{sec}] (TCH) Proposed x={round_array(x_cand)}, "
                                f"F_pred={round_array(Fp)}, "
                                f"G_pred={round_array(Gp) if Gp is not None else '[]'}")
                        else:
                            print(f"[{sec}] No eligible Tchebycheff candidate.")

                        # (2) Additional single-objective proposals (min f1, min f2)
                        #     only once every max_total_weights iterations.
                        if do_extra_single_obj:
                            # (2a) Additional single-objective proposal towards min f1
                            x_f1, Fp_f1, Gp_f1 = singleobj_propose_one(
                                0,
                                A["df_X"],
                                A["LB"], A["UB"], A["D"],
                                scip_exec=scip_exec,
                                feas_tol=feas_tol,
                                eps=eps,
                                sympy_exprs=sympy_exprs,
                                lambdas_f=lambdas_f,
                                lambdas_g=lambdas_g,
                                max_attempts=max_attempts_singleobj,
                                rng_seed=seed,
                            )
                            if x_f1 is not None:
                                proposals.append({
                                    "concept": sec,
                                    "x": x_f1,
                                    "F_pred": Fp_f1,
                                    "G_pred": (Gp_f1 if Gp_f1 is not None else np.zeros((0,))),
                                })
                                print(f"[{sec}] (SingleObj f1) Proposed x={round_array(x_f1)}, "
                                    f"F_pred={round_array(Fp_f1)}, "
                                    f"G_pred={round_array(Gp_f1) if Gp_f1 is not None else '[]'}")

                            # (2b) Additional single-objective proposal towards min f2
                            x_f2, Fp_f2, Gp_f2 = singleobj_propose_one(
                                1,
                                A["df_X"],
                                A["LB"], A["UB"], A["D"],
                                scip_exec=scip_exec,
                                feas_tol=feas_tol,
                                eps=eps,
                                sympy_exprs=sympy_exprs,
                                lambdas_f=lambdas_f,
                                lambdas_g=lambdas_g,
                                max_attempts=max_attempts_singleobj,
                                rng_seed=seed,
                            )
                            if x_f2 is not None:
                                proposals.append({
                                    "concept": sec,
                                    "x": x_f2,
                                    "F_pred": Fp_f2,
                                    "G_pred": (Gp_f2 if Gp_f2 is not None else np.zeros((0,))),
                                })
                                print(f"[{sec}] (SingleObj f2) Proposed x={round_array(x_f2)}, "
                                    f"F_pred={round_array(Fp_f2)}, "
                                    f"G_pred={round_array(Gp_f2) if Gp_f2 is not None else '[]'}")

                    # 3b) Handle no proposals => random fallback
                    if len(proposals) == 0:
                        archives, df_AF_global, df_AG_global, z_star, nadir = apply_random_fallback(
                            archives, no_max_samples, scip_exec,
                            feas_tol, eps, eps_dom, seed, alpha,
                        )
                        if df_AF_global is None:
                            break
                        continue

                    # 3c) Global DSS selection among proposals + true evaluation
                    archives, df_AF_global, df_AG_global, z_star, nadir = select_and_evaluate_proposal(
                        proposals, archives, df_AF_global, df_AG_global,
                        scip_exec, feas_tol, eps, eps_dom, seed, alpha,
                        z_star, nadir,
                    )

                    current_total = sum(len(A["df_X"]) for A in archives.values())
                    if current_total >= no_max_samples:
                        print(f"[Loop] Reached eval budget: {current_total}/{no_max_samples}. Stopping.")
                        break
            except KeyboardInterrupt:
                print("\n⚠️ Interrupted by user. Saving final results...")
            except Exception as e:
                print(f"\n❌ Error occurred: {e}")
                import traceback; traceback.print_exc()
            finally:
                print("\n💾 Saving final archives and saving plot...")
                try:
                    archives_to_save = archives_snapshot(archives)
                    with open(run_dir / "archives.pkl", "wb") as f:
                        pickle.dump(archives_to_save, f, protocol=pickle.HIGHEST_PROTOCOL)
                    if plot_flag:
                        plot_and_save_global_pareto(run_dir, archives, feas_tol, eps_dom, show=True)

                    final_total = sum(len(A["df_X"]) for A in archives.values())
                    print(f"\nRun complete. Final total evals = {final_total} (budget {no_max_samples}). "
                        f"Data saved to {run_dir}")
            
                except Exception as e2:
                    print(f"⚠️ Failed to save final results: {e2}")


# ====================================================
# Script entry point
# ====================================================

if __name__ == "__main__":
    all_problems = [
        # Unconstrained
        ("dtlz2",       problem_dtlz2),
        # ("zdt3",        problem_zdt3),
        # ("simulator",   problem_simulator),

        # Constrained
        # ("bnh",         problem_bnh),
        #("tnk",         problem_tnk),
        # ("welded_beam", problem_welded_beam),

    ]

    for name, prob in all_problems:
        print(f"\n======= Running problem: {name} =======")
        CONFIG = {
            "problem": prob,
            "scip_exec": r"C:\Users\z5653370\OneDrive - UNSW\Documents\scipoptsuite-10.0.0-win-x64\bin\scip",
            "seeds": list(range(1,2)),

            "lhs_per_dim": 10,
            "budget_per_dim": 15,

            # eps: proximity threshold + denominator safeguard
            "eps": 1e-6,
            # feas_tol: SCIP feasibility tolerance and all g<=0 checks
            "feas_tol": 1e-6,
            # eps_dom / alpha: still accepted but not critical with fixed ideal/nadir
            "eps_dom": 0.0,
            "alpha": 0.0,

            "max_attempts_singleobj": 30,
            "max_total_weights": 31,
            "pysr_iters": 60,
            "results_root": "Results_SR",
            "plot": True,
        }
        run_experiment(CONFIG)
